# Presentation Axis from VCTK

Build the presentation (gender) axis as a style delta that composes with
`with_deltas()`: extract styles for a balanced speaker subset of VCTK, fit a
direction over the 24 active `style_ttl` rows, validate it leave-speaker-out,
and listen to it.

**Why a corpus at all.** The obvious shortcut is to derive the axis from the
ten released presets. It was tried and it does not work. Restricted to the 24
active rows, PC1 over the ten presets explains 35.5% of the variance and does
not sort by gender -- M1 and M2 group with F1, F2 and F3. Leave-one-out
separation using the mean-difference direction reaches 7/10, against a 5/10
chance baseline: not a result. Ten points cannot determine a direction in 6,144
dimensions. That is the entire motivation for this notebook.

Phase 0's row-locality probe adds the shape of the problem: the M1 -> F1
difference is spread over roughly twenty scattered rows, and the strongest
single row reaches only about 0.30 travel. There is no gender bit to find -- it
is a direction, so it has to be fitted from enough speakers to be identifiable.

Same two-runtime split as
[docs/style_extraction_colab.ipynb](style_extraction_colab.ipynb): extraction is
torch on CUDA, playback is `onnxruntime` on CPU. Run this on a T4 or better.

## VCTK: get it, and check its terms yourself

- Source: <https://datashare.ed.ac.uk/handle/10283/3443>, DOI 10.7488/ds/2645.
- Around 10.9 GB, roughly 110 speakers. Confirm the current size and contents
  on the DataShare page -- this notebook does not verify them.
- Layout: `wav48_silence_trimmed/<spk>/<spk>_<utt>_mic{1,2}.flac`, mono 48 kHz,
  plus `speaker-info.txt` (id, age, gender, accent, region) and `txt/`.

**Licence: unresolved.** The Hugging Face card says CC-BY-4.0; other sources
say ODC-By 1.0. This notebook deliberately does not assert either. Read the
licence on the DataShare page before you use VCTK or anything derived from it,
and carry those terms onto the axis file this notebook emits.

**Age: do not use VCTK for an age axis.** Its speakers are 18-38 with the bulk
at 21-24. That range supports a presentation axis and identity-space structure;
it carries no age signal to learn from. The age axis needs different corpora
(see new-plan.md, phase 4).

Gender in `speaker-info.txt` is a recorded corpus label with two values. The
axis fitted here is the direction that separates those two labelled groups in
style space -- "presentation" in the sense the project means it, and nothing
more than that.

## The cost, and what it buys

Extraction is roughly 15 minutes per clip and does not get cheaper with shorter
clips. That number sets the design:

| Speakers per gender | Utterances each | Clips | Approx. GPU time |
|---|---|---|---|
| 5 | 1 | 10 | ~2.5 h |
| 8 | 1 | 16 | ~4 h |
| 10 | 1 | 20 | ~5 h |
| 10 | 2 | 40 | ~10 h |

The defaults below are 8 speakers per gender, one utterance each: 16 clips,
about four hours, which fits inside one Colab session with room to spare. 16
points is a real improvement on 10 presets but is still small for a 6,144-
dimensional fit -- expect the leave-speaker-out number to be noisy, and scale
up (more speakers before more utterances per speaker) once the pipeline is
proven. Every cell checkpoints per clip, so scaling up across several sessions
is just re-running the extraction cell.

**Validation is leave-speaker-out, never leave-clip-out.** Two clips from the
same speaker share that speaker's identity; holding one out while training on
the other measures nothing but leakage. Speakers are the unit.

A second utterance per speaker is still worth having eventually: it is the only
way to see how much of the fitted direction is utterance-specific rather than
speaker-specific.

In [ ]:
# 1. GPU check.
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No CUDA device. Runtime > Change runtime type > Hardware accelerator: '
        'GPU (T4 or better). Extraction needs it; only the playback cell is CPU.')
print('GPU: %s' % torch.cuda.get_device_name(0))
print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1024 ** 3))

In [ ]:
# 2. Drive. The HF cache is shared with the extraction notebook so WavLM-large
# (about 1.2 GB) is downloaded once across both.
import os
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

SHARED = Path('/content/drive/MyDrive/supertonic-style-extraction')
WORKSPACE = Path('/content/drive/MyDrive/supertonic-presentation-axis')
STYLES = WORKSPACE / 'vctk-styles'
LOGS = WORKSPACE / 'logs'
RESULTS = WORKSPACE / 'results'
for directory in (WORKSPACE, STYLES, LOGS, RESULTS):
    directory.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(SHARED / 'huggingface-cache')

MANIFEST = WORKSPACE / 'extraction_manifest.json'
print('Workspace: %s' % WORKSPACE)
print('HF_HOME:   %s' % os.environ['HF_HOME'])

In [ ]:
# 3. Extractor, dependencies, runtime checkout, and the shared module written
# by docs/style_extraction_colab.ipynb. The environment has to be rebuilt in
# every Colab VM; the logic is imported, not copied.
EXTRACTOR = Path('/content/supertonic.embed')
REPO = Path('/content/supertonic')
REPO_URL = 'https://github.com/supertone-inc/supertonic.git'   # or your fork

if not EXTRACTOR.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/kdrkdrkdr/supertonic.embed.git',
                    str(EXTRACTOR)], check=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)],
                   check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(EXTRACTOR / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'onnxruntime==1.23.1', 'numpy>=1.26.0', 'soundfile>=0.12.1',
                'librosa>=0.10.0', 'PyYAML>=6.0', 'huggingface_hub',
                'scikit-learn'], check=True)

STYLE_TOOLS_DIR = SHARED          # where notebook A wrote style_tools.py
if not (STYLE_TOOLS_DIR / 'style_tools.py').exists():
    raise FileNotFoundError(
        'style_tools.py not found in %s.\n'
        'Run docs/style_extraction_colab.ipynb once, through its '
        '"shared module" cell -- it writes style_tools.py to that Drive folder '
        'and this notebook imports it instead of duplicating the extraction, '
        'validation and blending code. If you keep it elsewhere, point '
        'STYLE_TOOLS_DIR at that folder.' % STYLE_TOOLS_DIR)
if str(STYLE_TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(STYLE_TOOLS_DIR))
import style_tools

ACTIVE_ROWS = style_tools.ACTIVE_ROWS
print('style_tools from %s' % style_tools.__file__)
print('Active rows (%d): %s' % (len(ACTIVE_ROWS), ACTIVE_ROWS))

In [ ]:
# 4. Model assets and the patched extractor config.
from huggingface_hub import snapshot_download

ASSETS = Path('/content/supertonic-assets')
snapshot_download(repo_id='Supertone/supertonic-3', local_dir=str(ASSETS),
                  allow_patterns=['onnx/*', 'voice_styles/*'])
ONNX_DIR = ASSETS / 'onnx'
PRESET_DIR = ASSETS / 'voice_styles'

TIMBRE_BATCH = None      # None keeps the extractor default (8); 1 is ~2.5 GB
CONFIG = style_tools.patch_extractor_config(
    src_config=EXTRACTOR / 'src' / 'config.yaml',
    out_config=WORKSPACE / 'extractor_config.yaml',
    onnx_dir=ONNX_DIR, presets_dir=PRESET_DIR, batch=TIMBRE_BATCH)
print('Presets: %s' % sorted(p.stem for p in PRESET_DIR.glob('*.json')))
print('Config:  %s' % CONFIG)

### Before the batch: run the smoke test

The version question from
[docs/style_extraction_colab.ipynb](style_extraction_colab.ipynb) applies here
unchanged -- `supertonic.embed` documents `Supertone/supertonic-2` weights and
we point it at `supertonic-3`. If you have not already run that notebook's
single-clip smoke test successfully, run it before starting a multi-hour VCTK
batch. The extraction cell below prints a reminder and stops after the first
clip if `SMOKE_TEST_FIRST` is left on.

In [ ]:
# 5. Locate VCTK. Preferred: an already-unpacked copy on Drive. Otherwise paste
# the download URL you got from the DataShare page after reading its licence.
VCTK_ROOT = Path('/content/drive/MyDrive/datasets/VCTK-Corpus-0.92')
VCTK_ARCHIVE_URL = ''        # only used if VCTK_ROOT does not exist
MIC = 'mic1'                 # VCTK ships mic1 and mic2 per utterance

if not VCTK_ROOT.exists():
    if not VCTK_ARCHIVE_URL:
        raise FileNotFoundError(
            'VCTK not found at %s.\n\n'
            'Go to https://datashare.ed.ac.uk/handle/10283/3443 (DOI '
            '10.7488/ds/2645), read the licence terms there -- they are '
            'reported inconsistently elsewhere and this notebook will not '
            'assert them for you -- then either:\n'
            '  (a) download and unpack the corpus into %s, or\n'
            '  (b) set VCTK_ARCHIVE_URL to the download link from that page '
            'and rerun this cell.\n'
            'The archive is around 10.9 GB, so (a) into Drive once is usually '
            'the better trade.' % (VCTK_ROOT, VCTK_ROOT))
    VCTK_ROOT.mkdir(parents=True, exist_ok=True)
    archive = Path('/content/vctk_download')
    print('Downloading (about 10.9 GB, expect a long wait)...')
    subprocess.run(['wget', '-c', '-O', str(archive), VCTK_ARCHIVE_URL], check=True)
    subprocess.run(['unzip', '-q', '-o', str(archive), '-d', str(VCTK_ROOT)],
                   check=True)

WAV_ROOT = VCTK_ROOT / 'wav48_silence_trimmed'
SPEAKER_INFO = VCTK_ROOT / 'speaker-info.txt'
for path in (WAV_ROOT, SPEAKER_INFO):
    if not path.exists():
        raise FileNotFoundError(
            '%s is missing. Expected the VCTK 0.92 layout: '
            'wav48_silence_trimmed/<spk>/<spk>_<utt>_mic{1,2}.flac plus '
            'speaker-info.txt. If your copy nests everything one level deeper, '
            'point VCTK_ROOT at that inner directory.' % path)

speaker_dirs = sorted(p.name for p in WAV_ROOT.iterdir() if p.is_dir())
print('VCTK root: %s' % VCTK_ROOT)
print('%d speaker directories, e.g. %s' % (len(speaker_dirs), speaker_dirs[:5]))

In [ ]:
# 6. Parse speaker-info.txt. Whitespace-separated: ID AGE GENDER ACCENT REGION.
import numpy as np

rows = []
for line in SPEAKER_INFO.read_text(errors='replace').splitlines():
    parts = line.split()
    if len(parts) < 3 or parts[0].upper() in ('ID', 'ID,'):
        continue
    speaker = parts[0] if parts[0].lower().startswith('p') or parts[0].lower().startswith('s') else 'p' + parts[0]
    try:
        age = int(parts[1])
    except ValueError:
        continue
    gender = parts[2].upper()
    if gender not in ('M', 'F'):
        continue
    rows.append({'speaker': speaker, 'age': age, 'gender': gender,
                 'accent': parts[3] if len(parts) > 3 else ''})

if not rows:
    raise ValueError('Parsed no speakers from %s -- check its format.' % SPEAKER_INFO)

available = set(speaker_dirs)
SPEAKERS = [r for r in rows if r['speaker'] in available]
ages = np.array([r['age'] for r in SPEAKERS])
counts = {g: sum(1 for r in SPEAKERS if r['gender'] == g) for g in ('M', 'F')}
print('%d speakers with audio (of %d in speaker-info.txt)'
      % (len(SPEAKERS), len(rows)))
print('Gender: M %d, F %d' % (counts['M'], counts['F']))
print('Age: min %d, max %d, median %d' % (ages.min(), ages.max(), int(np.median(ages))))
print('\nThat age range is why this corpus gives a presentation axis and not '
      'an age axis.')

In [ ]:
# 7. Balanced subset. Deterministic given the seed, so a disconnected session
# resumes onto exactly the same clip list.
import json
import random

SPEAKERS_PER_GENDER = 8
UTTERANCES_PER_SPEAKER = 1
SUBSET_SEED = 0
SKIP_UTTERANCES = {'001', '002'}   # the elicitation paragraph, read by everyone

rng = random.Random(SUBSET_SEED)


def utterances_for(speaker):
    files = sorted((WAV_ROOT / speaker).glob('%s_*_%s.flac' % (speaker, MIC)))
    return [f for f in files if f.stem.split('_')[1] not in SKIP_UTTERANCES]


eligible = {'M': [], 'F': []}
for record in sorted(SPEAKERS, key=lambda r: r['speaker']):
    if len(utterances_for(record['speaker'])) >= UTTERANCES_PER_SPEAKER:
        eligible[record['gender']].append(record)

SELECTED = []
for gender in ('M', 'F'):
    pool = eligible[gender]
    if len(pool) < SPEAKERS_PER_GENDER:
        raise ValueError('Only %d usable %s speakers with %s recordings; asked '
                         'for %d.' % (len(pool), gender, MIC, SPEAKERS_PER_GENDER))
    SELECTED.extend(rng.sample(pool, SPEAKERS_PER_GENDER))

CLIPS = []
for record in SELECTED:
    picks = utterances_for(record['speaker'])[:UTTERANCES_PER_SPEAKER]
    for path in picks:
        CLIPS.append({'path': path, 'name': path.stem, 'speaker': record['speaker'],
                      'gender': record['gender'], 'age': record['age']})

(WORKSPACE / 'subset.json').write_text(json.dumps(
    [dict(c, path=str(c['path'])) for c in CLIPS], indent=2))

print('%-10s %-7s %-5s %s' % ('speaker', 'gender', 'age', 'clip'))
for clip in CLIPS:
    print('%-10s %-7s %-5d %s' % (clip['speaker'], clip['gender'], clip['age'],
                                  clip['name']))
print('\n%d clips from %d speakers. Estimated %.1f h at 15 min/clip.'
      % (len(CLIPS), len(SELECTED), len(CLIPS) * 0.25))
print('Subset written to %s (deterministic given SUBSET_SEED).'
      % (WORKSPACE / 'subset.json'))

In [ ]:
# 8. Decode to mono WAV once, so the extraction step never has to care about
# FLAC. Native sample rate is preserved; nothing is resampled here.
import librosa
import soundfile as sf

PREPARED = WORKSPACE / 'prepared'
PREPARED.mkdir(exist_ok=True)

for clip in CLIPS:
    target = PREPARED / (clip['name'] + '.wav')
    if not target.exists():
        audio, sr = librosa.load(str(clip['path']), sr=None, mono=True)
        sf.write(str(target), audio, sr)
    clip['prepared'] = target

durations = [sf.info(str(c['prepared'])).duration for c in CLIPS]
print('Prepared %d clips in %s' % (len(CLIPS), PREPARED))
print('Durations: min %.2fs, median %.2fs, max %.2fs'
      % (min(durations), float(np.median(durations)), max(durations)))

In [ ]:
# 9. Extraction. Per-clip checkpoint on Drive, resume by skipping valid output,
# a failed clip is recorded and the batch continues.
import time
from datetime import datetime, timezone

SMOKE_TEST_FIRST = True   # stop after clip 1 so you can inspect it

manifest = json.loads(MANIFEST.read_text()) if MANIFEST.exists() else {}
pending = []
for clip in CLIPS:
    target = STYLES / (clip['name'] + '.json')
    if target.exists():
        try:
            if style_tools.validate_style_file(target, verbose=False)['ok']:
                manifest[clip['name']] = {'status': 'complete', 'path': str(target),
                                          'speaker': clip['speaker'],
                                          'gender': clip['gender']}
                continue
        except Exception:
            pass
        print('[redo] %s: checkpoint invalid' % clip['name'])
        target.unlink()
    pending.append(clip)

print('%d to extract, %d already done, estimated %.1f h.\n'
      % (len(pending), len(CLIPS) - len(pending), len(pending) * 0.25))

started_at = time.time()
done = 0
for clip in pending:
    manifest[clip['name']] = {'status': 'running', 'speaker': clip['speaker'],
                              'gender': clip['gender'],
                              'started_at': datetime.now(timezone.utc).isoformat()}
    MANIFEST.write_text(json.dumps(manifest, indent=2))
    try:
        produced = style_tools.run_extraction(
            clip_path=clip['prepared'], name=clip['name'], out_dir=STYLES,
            extractor_dir=EXTRACTOR, config_path=CONFIG,
            log_path=LOGS / ('extract_' + clip['name'] + '.log'))
        report = style_tools.validate_style_file(produced, verbose=True)
        manifest[clip['name']] = {
            'status': 'complete' if report['ok'] else 'suspect',
            'path': str(produced), 'speaker': clip['speaker'],
            'gender': clip['gender'], 'age': clip['age'],
            'completed_at': datetime.now(timezone.utc).isoformat(),
            'validation': report}
    except Exception as exc:
        manifest[clip['name']] = {'status': 'failed', 'speaker': clip['speaker'],
                                  'gender': clip['gender'], 'error': repr(exc)}
        print('[FAILED] %s: %r' % (clip['name'], exc))
    MANIFEST.write_text(json.dumps(manifest, indent=2))
    done += 1
    print('[progress] ' + style_tools.format_eta(done, len(pending),
                                                 time.time() - started_at) + '\n')
    if SMOKE_TEST_FIRST and done == 1:
        print('=' * 72)
        print('SMOKE TEST: stopped after one clip. Check the validation line '
              'above:\nshapes (50, 256) / (8, 16) and style_ttl row norms near '
              '1.0.\nIf it looks right, set SMOKE_TEST_FIRST = False and rerun '
              'this cell -- \nit resumes and skips this clip. If it does not, '
              'see the version\nnotes in docs/style_extraction_colab.ipynb '
              'before burning hours.')
        print('=' * 72)
        break

states = {}
for name, entry in manifest.items():
    states.setdefault(entry['status'], []).append(name)
for status in sorted(states):
    print('%-9s %d' % (status, len(states[status])))

In [ ]:
# 10. Load and validate every extracted style, then assemble the design matrix
# over the 24 active rows only (6,144 dims instead of 12,800).
DATA = []
problems = []
for clip in CLIPS:
    path = STYLES / (clip['name'] + '.json')
    if not path.exists():
        problems.append('%s: not extracted' % clip['name'])
        continue
    report = style_tools.validate_style_file(path, verbose=False)
    if not report['ok']:
        problems.append('%s: %s' % (clip['name'], report['problems']))
        continue
    ttl, dp = style_tools.load_style_arrays(path)
    DATA.append(dict(clip, ttl=ttl, dp=dp))

if problems:
    print('Excluded %d clip(s):' % len(problems))
    for problem in problems:
        print('  ! %s' % problem)

speakers = sorted({d['speaker'] for d in DATA})
by_gender = {g: sorted({d['speaker'] for d in DATA if d['gender'] == g})
             for g in ('M', 'F')}
print('\n%d clips, %d speakers (M %d, F %d).'
      % (len(DATA), len(speakers), len(by_gender['M']), len(by_gender['F'])))
if min(len(by_gender['M']), len(by_gender['F'])) < 3:
    raise RuntimeError('Fewer than 3 speakers in a class: nothing fittable yet. '
                       'Finish the extraction cell first.')

X = np.stack([d['ttl'][ACTIVE_ROWS].reshape(-1) for d in DATA]).astype(np.float64)
y = np.array([1 if d['gender'] == 'F' else 0 for d in DATA])
groups = np.array([d['speaker'] for d in DATA])
print('Design matrix: %s (clips x active-row values)' % (X.shape,))
norms = np.linalg.norm(np.stack([d['ttl'] for d in DATA]), axis=-1)
print('style_ttl row norms across the set: %.5f .. %.5f'
      % (norms.min(), norms.max()))

In [ ]:
# 11. Fit the direction, two ways.
#   mean-difference: mean(F) - mean(M), the same estimator the emotion deltas
#                    use, and the one that degenerates on ten presets.
#   LDA:             shrinkage LDA (Ledoit-Wolf), which uses the within-class
#                    covariance the mean difference ignores.
#
# There are far more dimensions (6,144) than clips, so LDA is fitted inside the
# subspace the clips actually span -- an economy SVD, at most n-1 components --
# and the resulting direction is mapped back. Fitting a 6,144 x 6,144 shrunk
# covariance directly is both ill-conditioned and slow, and buys nothing: the
# data has no support outside that subspace. In-sample LDA numbers are
# meaningless when p >> n (it can separate anything); only the
# leave-speaker-out cell below counts.
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

MAX_LDA_COMPONENTS = 32


def unit(vector):
    return vector / max(float(np.linalg.norm(vector)), 1e-12)


def fit_mean_difference(Xtr, ytr):
    return unit(Xtr[ytr == 1].mean(0) - Xtr[ytr == 0].mean(0))


def fit_lda(Xtr, ytr):
    center = Xtr.mean(0)
    centered = Xtr - center
    _, singular, Vt = np.linalg.svd(centered, full_matrices=False)
    keep = min(MAX_LDA_COMPONENTS, int((singular > singular.max() * 1e-8).sum()))
    basis = Vt[:keep]                       # (keep, 6144), orthonormal rows
    model = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    model.fit(centered @ basis.T, ytr)
    return unit(model.coef_.reshape(-1) @ basis)


FITTERS = {'mean_difference': fit_mean_difference, 'lda': fit_lda}
DIRECTIONS = {name: fitter(X, y) for name, fitter in FITTERS.items()}

for name, direction in DIRECTIONS.items():
    projection = X @ direction
    gap = projection[y == 1].mean() - projection[y == 0].mean()
    spread = np.sqrt(0.5 * (projection[y == 1].var(ddof=1)
                            + projection[y == 0].var(ddof=1)))
    print('%-16s in-sample class gap %.4f, pooled sd %.4f, d = %.2f'
          % (name, gap, spread, gap / max(spread, 1e-12)))

cosine = float(DIRECTIONS['mean_difference'] @ DIRECTIONS['lda'])
print('\ncos(mean_difference, lda) = %.3f' % cosine)

# Where does the direction live? Per-row energy over the 24 active rows.
energy = np.linalg.norm(DIRECTIONS['mean_difference'].reshape(len(ACTIVE_ROWS), 256), axis=1) ** 2
order = np.argsort(energy)[::-1]
print('\nTop rows by share of the mean-difference direction:')
for index in order[:8]:
    print('  row %2d: %5.1f%%' % (ACTIVE_ROWS[index], 100 * energy[index] / energy.sum()))
print('(Phase 0 found the M1->F1 difference spread over roughly twenty rows. '
      'A direction concentrated in one or two rows here would be a red flag, '
      'not a discovery.)')

In [ ]:
# 12. Leave-speaker-out validation. Every clip from a speaker is held out
# together: clips from the same speaker share identity, so leave-clip-out
# measures leakage rather than generalization.
def leave_speaker_out(X, y, groups, fitter):
    predictions = np.zeros(len(y), dtype=int)
    for speaker in sorted(set(groups)):
        test = groups == speaker
        train = ~test
        if len(set(y[train])) < 2:
            raise ValueError('Holding out %s leaves one class.' % speaker)
        direction = fitter(X[train], y[train])
        projection_train = X[train] @ direction
        threshold = 0.5 * (projection_train[y[train] == 1].mean()
                           + projection_train[y[train] == 0].mean())
        predictions[test] = ((X[test] @ direction) > threshold).astype(int)
    return predictions


majority = max(float((y == 1).mean()), float((y == 0).mean()))
RESULTS_SUMMARY = {'n_clips': int(len(y)), 'n_speakers': int(len(set(groups))),
                   'n_female_clips': int((y == 1).sum()),
                   'n_male_clips': int((y == 0).sum()),
                   'chance_majority': majority, 'chance_balanced': 0.5,
                   'methods': {}}

for name, fitter in FITTERS.items():
    predictions = leave_speaker_out(X, y, groups, fitter)
    clip_accuracy = float((predictions == y).mean())
    speaker_correct = []
    for speaker in sorted(set(groups)):
        mask = groups == speaker
        speaker_correct.append(float((predictions[mask] == y[mask]).mean()) > 0.5)
    speaker_accuracy = float(np.mean(speaker_correct))
    RESULTS_SUMMARY['methods'][name] = {
        'loso_clip_accuracy': clip_accuracy,
        'loso_speaker_accuracy': speaker_accuracy,
        'correct_clips': int((predictions == y).sum())}
    print('%-16s leave-speaker-out: %d/%d clips (%.1f%%), %.1f%% of speakers'
          % (name, (predictions == y).sum(), len(y), 100 * clip_accuracy,
             100 * speaker_accuracy))

print('\nChance: %.1f%% balanced, %.1f%% majority-class.'
      % (50.0, 100 * majority))
print('For comparison, the same estimator over the ten released presets reaches '
      '7/10 leave-one-out against a 5/10 baseline -- the result this notebook '
      'exists to beat. Below about 80% here, treat the axis as unfitted and '
      'add speakers before spending time listening to it.')

(RESULTS / 'separation.json').write_text(json.dumps(RESULTS_SUMMARY, indent=2))
print('\nWrote %s' % (RESULTS / 'separation.json'))

In [ ]:
# 13. Emit the axis as a style delta in the repo's JSON schema, so it drops
# straight into with_deltas().
#
# Scale: the unit direction is multiplied by the observed class gap, so
# weight +1.0 is one full male-mean -> female-mean shift and -1.0 is the
# reverse. Sign convention: positive moves toward the group VCTK labels F.
# Inactive rows are exactly zero -- the axis is defined only where the presets
# actually vary -- and style_dp is zero, so duration is untouched unless
# someone deliberately passes include_duration.
AXIS_METHOD = 'mean_difference'      # or 'lda'

direction = DIRECTIONS[AXIS_METHOD]
projection = X @ direction
scale = float(projection[y == 1].mean() - projection[y == 0].mean())

delta_ttl = np.zeros((50, 256), dtype=np.float32)
delta_ttl[ACTIVE_ROWS] = (direction * scale).reshape(len(ACTIVE_ROWS), 256)
delta_dp = np.zeros((8, 16), dtype=np.float32)

axis_path = RESULTS / 'presentation.json'
style_tools.write_style_json(axis_path, delta_ttl, delta_dp, metadata={
    'kind': 'style_difference',
    'axis': 'presentation',
    'sign': 'positive weight moves toward the group labelled F in VCTK '
            'speaker-info.txt',
    'method': AXIS_METHOD,
    'scale': scale,
    'scale_meaning': 'weight 1.0 equals one class-mean separation',
    'active_rows': ACTIVE_ROWS,
    'inactive_rows_are_zero': True,
    'style_dp_is_zero': True,
    'corpus': 'VCTK 0.92 (https://datashare.ed.ac.uk/handle/10283/3443, DOI '
              '10.7488/ds/2645) -- confirm licence terms at that page before '
              'redistributing anything derived from it',
    'mic': MIC,
    'speakers': sorted(set(groups.tolist())),
    'clips': [d['name'] for d in DATA],
    'validation': RESULTS_SUMMARY,
})
print('Wrote %s (%.1f KB)' % (axis_path, axis_path.stat().st_size / 1024))
print('Delta magnitude: %.4f over %d active rows, %.1f%% of rows zero'
      % (np.linalg.norm(delta_ttl), len(ACTIVE_ROWS),
         100 * (1 - len(ACTIVE_ROWS) / 50)))

check_ttl, check_dp = style_tools.load_style_arrays(axis_path)
assert check_ttl.shape == (50, 256) and check_dp.shape == (8, 16)
assert np.allclose(check_ttl, delta_ttl, atol=1e-6)
print('Round-trips through the loader unchanged.')

## Listen to the axis

A separation number says the two labelled groups are linearly distinguishable
in style space. It does not say the direction is audible, that it moves
presentation rather than some correlated recording artefact, or that it stays
intelligible off the ends. That is a listening question.

The cell below applies the axis to a released preset at several weights, using
the same accumulate-then-normalize-once rule as `Style.with_deltas()`: per row,
over the last axis, exactly once. It checks that weight 0 is an exact identity
and that per-row norms survive the blend, cross-checks against the repo's own
`with_deltas()` when the fork is the checkout in use, then synthesizes on CPU
with numpy re-seeded before each render so the clips differ only by the style
tensor.

Post the resulting clips into the conversation with their manifest lines --
endpoints and midpoint at minimum -- and build the listening bench artifact the
project uses for verdicts. Files in Drive are unhearable.

In [ ]:
# 14. Apply, verify, synthesize.
import soundfile as sf
from IPython.display import Audio, display

sys.path.insert(0, str(REPO / 'py'))
import helper

for attr in ('Style', 'load_text_to_speech', 'load_voice_style'):
    if not hasattr(helper, attr):
        raise AttributeError(
            'py/helper.py in %s has no %r. REPO_URL points at a checkout that '
            'does not match this fork.' % (REPO, attr))

BASE_VOICES = ['M1', 'F1']
WEIGHTS = [-1.0, -0.5, 0.0, 0.5, 1.0]
TEXT = 'The quick brown fox jumps over the lazy dog.'
LANG = 'en'
TOTAL_STEP = 8
SPEED = 1.05
SEED = 0

text_to_speech = helper.load_text_to_speech(str(ONNX_DIR))    # CPU
audio_dir = RESULTS / 'listening_set'
audio_dir.mkdir(parents=True, exist_ok=True)
has_repo_blend = hasattr(helper, 'Style') and hasattr(helper.Style, 'with_deltas')
print('Repo with_deltas() available for cross-check: %s\n' % has_repo_blend)

manifest_lines = []
for base_name in BASE_VOICES:
    base_path = PRESET_DIR / (base_name + '.json')
    base_ttl, base_dp = style_tools.load_style_arrays(base_path)
    base_norms = np.linalg.norm(base_ttl, axis=-1)

    for weight in WEIGHTS:
        ttl, dp = style_tools.with_deltas_np(
            base_ttl, base_dp, [(delta_ttl, delta_dp, weight)],
            include_duration=False)

        if weight == 0.0:
            assert np.array_equal(ttl, base_ttl), 'zero weight must be identity'
        new_norms = np.linalg.norm(ttl, axis=-1)
        assert np.abs(new_norms - base_norms).max() < 1e-4, 'row norms drifted'

        if has_repo_blend:
            repo_style = helper.load_voice_style([str(base_path)])
            repo_delta = helper.Style(delta_ttl[None, ...], delta_dp[None, ...])
            blended = repo_style.with_deltas([(repo_delta, weight)])
            drift = float(np.abs(blended.ttl[0] - ttl).max())
            assert drift < 1e-5, 'notebook blend disagrees with helper.py (%g)' % drift

        style = helper.Style(ttl[None, ...].astype(np.float32),
                             dp[None, ...].astype(np.float32))
        np.random.seed(SEED)
        wav, duration = text_to_speech(TEXT, LANG, style, TOTAL_STEP, SPEED)
        trimmed = wav[0, :int(text_to_speech.sample_rate * duration[0].item())]

        tag = ('%+.2f' % weight).replace('.', 'p').replace('+', 'pos').replace('-', 'neg')
        out_path = audio_dir / ('presentation_%s_w%s.wav' % (base_name, tag))
        sf.write(str(out_path), trimmed, text_to_speech.sample_rate)
        line = ('%s | base=%s | axis=presentation (%s, VCTK %d speakers) | '
                'weight=%+.2f | text=%r | total_step=%d speed=%.2f seed=%d '
                'lang=%s include_duration=False'
                % (out_path.name, base_name, AXIS_METHOD, len(set(groups)),
                   weight, TEXT, TOTAL_STEP, SPEED, SEED, LANG))
        manifest_lines.append(line)
        print(line)
        display(Audio(str(out_path)))

(RESULTS / 'listening_set_manifest.txt').write_text('\n'.join(manifest_lines) + '\n')
print('\nWrote %s' % (RESULTS / 'listening_set_manifest.txt'))

## What to listen for, and what to do next

Judge, in this order:

1. **Weight 0 is the untouched preset.** It is asserted above; if it does not
   sound identical to the plain voice, something upstream is wrong.
2. **Monotonicity.** Does -1.0 -> +1.0 move in one consistent direction, or
   does it wander? A direction that only works at one end is not an axis.
3. **What actually moved.** Presentation, or just pitch? Or channel character
   picked up from VCTK's recording conditions? A mean difference over two
   speaker groups absorbs everything that differs between those groups, and
   with 16 speakers that includes accent and room.
4. **Where it breaks.** Push past +/-1.0. Weights are deliberately unclamped;
   find the point where intelligibility goes, and record it.
5. **Identity retention.** M1 at +1.0 should still be recognizably a variation
   of M1, not a different speaker.

If it holds up:

- Scale the corpus. More speakers first, then a second utterance per speaker to
  separate speaker-specific from utterance-specific structure. The extraction
  cell resumes, so this is several sessions, not a restart.
- Re-fit with LDA once there are enough speakers for the within-class
  covariance to mean something, and compare directions by cosine.
- Copy `presentation.json` into the repo's emotion/axis asset directory (these
  files are git-ignored -- do not commit them) and drive it through
  `with_deltas()` alongside an emotion delta to check that composition behaves.
- Measure rather than only listen: the parselmouth loop in new-plan.md is what
  turns "sounds like it moved" into a monotonicity curve.

Open items this notebook does not resolve:

- VCTK's licence. Confirm on the DataShare page; the sources disagree.
- Whether `supertonic.embed` (documented against `Supertone/supertonic-2`) is
  truly compatible with the v3 assets. The smoke test says the shapes and norms
  are right, which is necessary, not sufficient. The listening cell is the real
  check.
- Whether VCTK's studio conditions transfer to other recording conditions at
  all. The axis is fitted where the corpus lives.

In [ ]:
# 15. Download the axis, its metrics and the listening set.
import shutil

from google.colab import files

bundle = Path('/content/presentation-axis')
if bundle.exists():
    shutil.rmtree(bundle)
shutil.copytree(str(RESULTS), str(bundle))
shutil.copy2(str(WORKSPACE / 'subset.json'), str(bundle / 'subset.json'))
if MANIFEST.exists():
    shutil.copy2(str(MANIFEST), str(bundle / 'extraction_manifest.json'))

archive = shutil.make_archive('/content/presentation-axis', 'zip', str(bundle))
print('Archive: %s (%.1f MB)' % (archive, Path(archive).stat().st_size / 1e6))
print('Drive copy: %s' % RESULTS)
files.download(archive)